[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/28_moe.ipynb)

# 🔴 Hard: Mixture of Experts (MoE)

Implement a **Mixture of Experts** layer (Mixtral / Switch Transformer style).

### Signature
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn

In [1]:
# ✏️ YOUR IMPLEMENTATION HERE

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        self.d_model=d_model
        self.df_ff=d_ff
        self.num_experts= num_experts
        self.top_k=top_k
        self.router=nn.Linear(d_model, num_experts)
        self.experts=nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.ReLU(),
                nn.Linear(d_model, d_ff)
            )
        ])
    def forward(self, x):
        B,S, _ = x.shape
        gates=self.router(x)
        probs=torch.softmax(gates, dim=-1)
        scores, indices= torch.topk(probs, self.top_k, dim=-1)

        self.expert_outputs= torch.zeros(B,S,d_model, device=x.device)

        for i in range(self.top_k):
          expert_indice= indices[:,:,i]
          score= scores[:, :, i]

          for e in range(self.num_experts):
            mask=(experts==e)
            if mask.any():
              out=self.experts[e](x)
              expert_outputs+= mask * score * out
        return expert_outputs





NameError: name 'nn' is not defined

In [ ]:
# 🧪 Debug
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('Output:', moe(x).shape)
print('Params:', sum(p.numel() for p in moe.parameters()))

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('moe')

In [ ]:
# ── SETUP ────────────────────────────────────────────────────────
# batch=1, seq=2, d_model=4, d_ff=8, num_experts=2, top_k=1
# x shape: (1, 2, 4)

x = [[[0.1, 0.2, 0.6, 0.3],   # token 0
      [0.3, 0.4, 0.1, 0.2]]]  # token 1

# ── LINE 1: batch, seq, _ = x.shape ──────────────────────────────
batch = 1
seq   = 2
_     = 4   # d_model, ignored since we use self.d_model

# ── LINE 2: gates = self.router(x) ───────────────────────────────
# router = nn.Linear(4, 2)  →  projects each token from d_model=4 to num_experts=2
# assume router weights W shape (2,4) and bias b shape (2,):
# W = [[0.1, 0.2, 0.3, 0.4],   # expert 0 direction
#      [0.4, 0.3, 0.2, 0.1]]   # expert 1 direction
# b = [0.0, 0.0]

# token 0: [0.1, 0.2, 0.6, 0.3]
# expert 0 score: 0.1*0.1 + 0.2*0.2 + 0.6*0.3 + 0.3*0.4 = 0.01+0.04+0.18+0.12 = 0.35
# expert 1 score: 0.1*0.4 + 0.2*0.3 + 0.6*0.2 + 0.3*0.1 = 0.04+0.06+0.12+0.03 = 0.25

# token 1: [0.3, 0.4, 0.1, 0.2]
# expert 0 score: 0.3*0.1 + 0.4*0.2 + 0.1*0.3 + 0.2*0.4 = 0.03+0.08+0.03+0.08 = 0.22
# expert 1 score: 0.3*0.4 + 0.4*0.3 + 0.1*0.2 + 0.2*0.1 = 0.12+0.12+0.02+0.02 = 0.28

gates = [[[0.35, 0.25],   # token 0: raw scores for expert 0 and 1
          [0.22, 0.28]]]  # token 1: raw scores for expert 0 and 1
# shape: (1, 2, 2) = (batch, seq, num_experts)

# ── LINE 3: probs = softmax(gates, dim=-1) ───────────────────────
# token 0: softmax([0.35, 0.25])
#   e^0.35 = 1.419,  e^0.25 = 1.284,  sum = 2.703
#   expert 0: 1.419/2.703 = 0.525
#   expert 1: 1.284/2.703 = 0.475

# token 1: softmax([0.22, 0.28])
#   e^0.22 = 1.246,  e^0.28 = 1.323,  sum = 2.569
#   expert 0: 1.246/2.569 = 0.485
#   expert 1: 1.323/2.569 = 0.515

probs = [[[0.525, 0.475],   # token 0 — expert 0 wins slightly
          [0.485, 0.515]]]  # token 1 — expert 1 wins slightly
# shape: (1, 2, 2)

# ── LINE 4: scores, indices = topk(probs, k=1) ───────────────────
# top_k=1 so just keep the single best expert per token

scores  = [[[0.525],   # token 0: best score
            [0.515]]]  # token 1: best score
# shape: (1, 2, 1)

indices = [[[0],   # token 0 chose expert 0
            [1]]]  # token 1 chose expert 1
# shape: (1, 2, 1)

# ── LINE 5: expert_outputs = zeros ───────────────────────────────
expert_outputs = [[[0.0, 0.0, 0.0, 0.0],   # token 0
                   [0.0, 0.0, 0.0, 0.0]]]  # token 1
# shape: (1, 2, 4)

# ══════════════════════════════════════════════════════════════════
# OUTER LOOP i=0  (only 1 iteration since top_k=1)
# ══════════════════════════════════════════════════════════════════

# ── LINE 6: expert_idx = indices[:,:,0] ──────────────────────────
expert_idx = [[0, 1]]   # token 0 → expert 0, token 1 → expert 1
# shape: (1, 2)

# ── LINE 7: score = scores[:,:,0] ────────────────────────────────
score = [[0.525, 0.515]]   # trust scores
# shape: (1, 2)

# ══════════════════════════════════════════════════════════════════
# INNER LOOP e=0  (processing expert 0)
# ══════════════════════════════════════════════════════════════════

# ── LINE 8: mask = (expert_idx == 0) ─────────────────────────────
mask = [[True, False]]   # only token 0 chose expert 0
# shape: (1, 2)

# ── LINE 9: mask.any() = True → proceed ──────────────────────────

# ── LINE 10: out = self.experts[0](x) ────────────────────────────
# expert 0 is nn.Linear(4,8) → ReLU → nn.Linear(8,4)
# runs on BOTH tokens but we'll only keep token 0's result
# assume expert 0 output:
out = [[[0.4, 0.8, 0.3, 0.6],   # token 0 through expert 0
        [0.2, 0.5, 0.7, 0.1]]]  # token 1 through expert 0 (will be zeroed)
# shape: (1, 2, 4)

# ── LINE 11: expert_outputs += mask * score * out ────────────────
# mask.unsqueeze(-1)  = [[[True ], [False]]]   shape (1,2,1)
# score.unsqueeze(-1) = [[[0.525], [0.515]]]   shape (1,2,1)

# token 0: True  * 0.525 * [0.4, 0.8, 0.3, 0.6] = [0.210, 0.420, 0.158, 0.315]  ✓
# token 1: False * 0.515 * [0.2, 0.5, 0.7, 0.1] = [0.000, 0.000, 0.000, 0.000]  ✗

expert_outputs = [[[0.210, 0.420, 0.158, 0.315],   # token 0 updated!
                   [0.000, 0.000, 0.000, 0.000]]]  # token 1 still zero

# ══════════════════════════════════════════════════════════════════
# INNER LOOP e=1  (processing expert 1)
# ══════════════════════════════════════════════════════════════════

# ── LINE 8: mask = (expert_idx == 1) ─────────────────────────────
mask = [[False, True]]   # only token 1 chose expert 1

# ── LINE 10: out = self.experts[1](x) ────────────────────────────
out = [[[0.5, 0.3, 0.6, 0.2],   # token 0 through expert 1 (will be zeroed)
        [0.7, 0.4, 0.2, 0.8]]]  # token 1 through expert 1
# shape: (1, 2, 4)

# ── LINE 11: expert_outputs += mask * score * out ────────────────
# token 0: False * 0.525 * [0.5,0.3,0.6,0.2] = [0.000, 0.000, 0.000, 0.000]  ✗
# token 1: True  * 0.515 * [0.7,0.4,0.2,0.8] = [0.361, 0.206, 0.103, 0.412]  ✓

expert_outputs = [[[0.210, 0.420, 0.158, 0.315],   # token 0 — from expert 0
                   [0.361, 0.206, 0.103, 0.412]]]  # token 1 — from expert 1

# ── RETURN ───────────────────────────────────────────────────────
# shape: (1, 2, 4) — same as input!
# token 0 was processed entirely by expert 0 (score 0.525)
# token 1 was processed entirely by expert 1 (score 0.515)